In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn.linear_model as lm
import pandas as pd
import cloudpickle

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

import mat73

In [ ]:
my_dict = mat73.loadmat('SingleRegion_Aggression_data.mat')

In [ ]:
TrainsetMouse = my_dict['TrainsetMouse']
mbeh_all2 = my_dict['mbeh_all2']
mcond_all2 = my_dict['mcond_all2']
mouse_all2 = my_dict['mouse_all2']
mpow_all2 = my_dict['mpow_all2']
mpow_all3s = my_dict['mpow_all3s']
mtimecondbeh2 = my_dict['mtimecondbeh2']
mu_all3 = my_dict['mu_all3']
testsetMouse = my_dict['testsetMouse']


In [ ]:
idxs_pos = (mcond_all2==4)&(mbeh_all2==1)
idx_neg = ((mcond_all2==6)&(mbeh_all2==2))|((mcond_all2==8)&(mbeh_all2==2))
selection_indices = idxs_pos|idx_neg
print(np.mean(idxs_pos))
print(np.mean(idx_neg))
print(np.mean(selection_indices))

In [ ]:
y = np.zeros(N_samples)
y[idxs_pos] = 1
mpower_reduced = mpow_all2[:,:,selection_indices]
y_reduced = y[selection_indices]
train_idxs_reduced = train_idxs[selection_indices]
y_train = y_reduced[train_idxs_reduced==1]
y_test = y_reduced[train_idxs_reduced==0]
m_idx_unique_test = np.unique(mouse_idxs[train_idxs==0])
mouse_idxs_reduced = mouse_idxs[selection_indices]
m_test = mouse_idxs_reduced[train_idxs_reduced==0]

In [ ]:
np.unique(mbeh_all2)

In [ ]:

model_list = []
test_aucs = np.zeros(11)

test_aucs_mouse = np.zeros((11,9))

for i in range(11):
    XT = np.squeeze(mpower_reduced[:,i,:])
    X = np.transpose(XT)
    X = X*10
    X[X>6] = 6
    Xtrain = X[train_idxs_reduced==1]
    Xtest = X[train_idxs_reduced==0]
    model = LogisticRegression(max_iter=1000)
    model.train(Xtrain,y_train)
    test_aucs[i] = roc_auc_score(y_test,model.decision_function(Xtest))
    model_list.append(model)
    for j in range(9):
        test_aucs_mouse[i,j] = roc_auc_score(y_test[m_test==20+j],
                                np.squeeze(S_test[m_test==20+j,0]*model.Phi))
    print(i,test_aucs[i])

In [ ]:
region_list = ['IL','LHb','LSN','MDThal','MeA','NAc','OFC','PL','V1','VHipp','VMHvl']
for i in range(11):
    print('Region ',region_list[i],test_aucs[i])

In [ ]:
import cloudpickle

In [ ]:
myDict = {'models':model_list}
with open('Comparison4.p','wb') as f:
    cloudpickle.dump(myDict,f)
np.savetxt('Comparison4.csv',test_aucs_mouse,fmt='%0.8f',delimiter=',')